In [2]:
import pandas as pd
import numpy as np

In [3]:
rucc = pd.read_csv(r"C:\Users\manju\OneDrive\Desktop\Documents\Data Analytics\DA Semester 4\Capstone 2\Supplementary Data\Old Files\rural_urban_codes.csv",encoding='latin1')

In [4]:
rucc_fips = rucc[rucc["Attribute"] == "RUCC_2023"][
    ["FIPS", "State", "County_Name"]
].copy()

rucc_fips["fips"] = rucc_fips["FIPS"].astype(str).str.zfill(5)


In [7]:
hospital = pd.read_csv(r"C:\Users\manju\OneDrive\Desktop\Documents\Data Analytics\DA Semester 4\Capstone 2\ML Modelling\hospital_kpi_ready_with_rucc.csv")

In [8]:
import re

def clean_county(x):
    x = str(x).upper().strip()
    x = re.sub(r"\s+(COUNTY|MUNICIPIO|PARISH|BOROUGH|CENSUS AREA|ISLAND|CITY)$", "", x)
    x = x.replace("SAINT ", "ST ")
    x = re.sub(r"[^\w\s]", "", x)
    return x.strip()

hospital["county_key"] = hospital["county"].apply(clean_county)
rucc_fips["county_key"] = rucc_fips["County_Name"].apply(clean_county)


In [9]:
hospital = hospital.merge(
    rucc_fips[["State", "county_key", "fips"]],
    left_on=["state_code", "county_key"],
    right_on=["State", "county_key"],
    how="left"
)
hospital.drop(columns=["county_key", "State"], inplace=True)

In [10]:
hospital["fips"].isna().mean()

np.float64(0.0859628311585607)

In [11]:
import pandas as pd

census = pd.read_csv(
    r"C:\Users\manju\OneDrive\Desktop\Documents\Data Analytics\DA Semester 4\Capstone 2\Supplementary Data\census_ml.csv"
)
census["fips"] = census["fips"].astype(str).str.zfill(5)

hospital = hospital.merge(
    census,
    on=["fips", "year"],
    how="left"
)


# Sanity check
print("Rows after merge:", hospital.shape[0])
print(hospital[["population", "median_income"]].isna().mean())

Rows after merge: 75870
population       0.164347
median_income    0.164360
dtype: float64


In [13]:
# Keep CMS county name (optional, for readability)
hospital.rename(columns={"county_x": "county"}, inplace=True)

# Drop Census county column
hospital.drop(columns=["county_y", "state"], inplace=True)


In [14]:
hospital.head()

,year,state_code,provider_type,ccn_facility_type,number_of_beds,total_bed_days_available,occupancy_rate,total_discharges__v___xviii___xix___unknown_,total_days__v___xviii___xix___unknown_,fte___employees_on_payroll,...,uncompensated_care_percent,charity_care_percent,rucc_code,rural_urban,fips,population,median_age,median_income,poverty_rate,higher_education_rate
0,2011,AL,General Short-Term (includes CAH),Short-Term Hospital,114.0,41610.0,0.472026,5283.0,19641.0,598.72,...,2.482716,2.263904,4.0,Rural,01095,NaN,NaN,NaN,NaN,NaN
1,2011,MT,General Short-Term (includes CAH),Critical Access Hospital,25.0,9125.0,0.131068,155.0,1196.0,70.56,...,9.372838,2.320308,3.0,Urban,30009,NaN,NaN,NaN,NaN,NaN
2,2011,AL,General Short-Term (includes CAH),Short-Term Hospital,46.0,16790.0,0.159678,892.0,2681.0,72.65,...,15.092862,10.117954,6.0,Rural,01123,NaN,NaN,NaN,NaN,NaN
3,2011,AL,Rehabilitation Hospital,Rehabilitation Hospital,100.0,36500.0,0.887068,2390.0,32378.0,297.77,...,-0.031671,4.613961,1.0,Urban,01073,NaN,NaN,NaN,NaN,NaN
4,2011,FL,Rehabilitation Hospital,Rehabilitation Hospital,70.0,25550.0,0.663836,1370.0,16961.0,154.60,...,-0.090532,7.756714,1.0,Urban,12103,NaN,NaN,NaN,NaN,NaN


In [17]:
hospital[["population", "median_income", "poverty_rate"]].isna().sum()

population       12469
median_income    12470
poverty_rate     12470
dtype: int64

In [ ]:
# Save the final dataset
hospital.to_csv(
    r"C:\Users\manju\OneDrive\Desktop\Documents\Data Analytics\DA Semester 4\Capstone 2\ML Modelling\hospital_kpi_cms_census_rucc.csv",
    index=False,
)